# Potato Leaf Disease Classification

## Objective
Build a CNN model to classify potato leaf diseases from leaf images.

## Dataset
Source: [Potato Leaf Disease Dataset](https://www.kaggle.com/datasets/muhammadardiputra/potato-leaf-disease-dataset)

Classes:
- Potato Early Blight
- Potato Late Blight
- Potato Healthy

## Workflow
1. Download dataset
2. Clean corrupted images
3. Create TensorFlow datasets
4. Perform EDA and visualization
5. Train CNN model
6. Evaluate performance
7. Run inference

# Importing libraries

## Hyperparameters

| Parameter | Value |
|------------|---------|
| IMAGE_SIZE | 256 |
| CHANNELS | 3 |
| BATCH_SIZE | 32 |
| EPOCHS | 15 |

## Libraries

In [ ]:
# Utilities for storage and fetching dataset
import joblib
import kagglehub
import numpy as np
import os
from pathlib import Path
import shutil

# Deep learning model and model evaluation
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2 # type: ignore
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input # type: ignore
from tensorflow.keras import layers, models # type: ignore
from sklearn.metrics import confusion_matrix, classification_report

# Visualization
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Define project paths
dataset_dir = Path("../data")
dataset_dir.mkdir(exist_ok=True)

model_dir = Path("../model")
model_dir.mkdir(exist_ok=True)

model_path = model_dir / "model.keras"
status_path = model_dir / "model_stats.pkl"
variables_path = model_dir / "variables.pkl"

# Configure training parameters
BATCH_SIZE=32
IMAGE_SIZE=256
CHANNEL=3
EPOCHS=15

# Fetching the dataset

## Dataset Information

- Total classes: 3
- Image format: JPG
- Train/Validation/Test split
- Image size used: 256×256

- Dataset link: [Potato Leaf Disease Dataset
](https://www.kaggle.com/datasets/muhammadardiputra/potato-leaf-disease-dataset)

In [ ]:
# Download and organize the dataset

print("Downloading the dataset")
path = kagglehub.dataset_download("muhammadardiputra/potato-leaf-disease-dataset")
print(f"Dataset downloaded to the path: {path}")
shutil.copytree(path, dataset_dir, dirs_exist_ok=True)
print(f"Dataset copied to the path: {dataset_dir}")

# Define dataset split directories

training_images = dataset_dir / "Potato" /"Train"
testing_images = dataset_dir / "Potato" / "Test"
validation_images = dataset_dir / "Potato" / "Valid"

## Removing corrupted images before data preprocessing

In [ ]:
# Detect and remove corrupted image files

corrupted_images = 0

for file in Path(dataset_dir).rglob("*"):
    if file.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        try:
            img = tf.io.read_file(str(file))
            tf.image.decode_image(img)
        except Exception as e:
            os.remove(file)
            print(f"Deleted: {file}")
            corrupted_images+=1

print(f"\nNo. of corrupted images removed: {corrupted_images}")

## Dividing the data into batches for better performance

In [ ]:
# Create batched datasets for training, testing, and validation

print("Generating batches for training data: ")
batch_train = tf.keras.preprocessing.image_dataset_from_directory(
    directory=training_images,
    seed=42,
    shuffle=True,
    image_size=[IMAGE_SIZE, IMAGE_SIZE],
    batch_size=BATCH_SIZE
)

print("\n\nGenerating batches for testing data: ")
batch_test = tf.keras.preprocessing.image_dataset_from_directory(
    directory=testing_images,
    seed=42,
    shuffle=True,
    image_size=[IMAGE_SIZE, IMAGE_SIZE],
    batch_size=BATCH_SIZE
)

print("\n\nGenerating batches for validation data: ")
batch_valid = tf.keras.preprocessing.image_dataset_from_directory(
    directory=validation_images,
    seed=42,
    shuffle=True,
    image_size=[IMAGE_SIZE, IMAGE_SIZE],
    batch_size=BATCH_SIZE
)

# Extract class labels

class_names = batch_test.class_names

In [ ]:
# Inspect a sample batch of images and labels

for batch_size, label_size in batch_train.take(1):
    print(f"Image shape: {batch_size.shape}")
    print(f"Image size: ", {label_size.numpy})

## Visualizing the images

In [ ]:
for images, labels in batch_train.take(1):
    titles = [class_names[int(labels[i])] for i in range(12)]

    fig = make_subplots(
        rows=3,
        cols=4,
        subplot_titles=titles
    )

    for i in range(12):
        row = i // 4 + 1
        col = i % 4 + 1

        fig.add_trace(
            go.Image(z=images[i].numpy().astype("uint8")),
            row=row,
            col=col
        )

    fig.update_layout(
        height=900,
        width=1200,
        showlegend=False
    )

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)

    fig.show()

## Dataset batch distribution

In [ ]:
train_len = len(batch_train)
test_len = len(batch_test)
validation_len = len(batch_valid)

total = train_len + test_len + validation_len
print(f"Total number of batches: {total}")
print(f"Total number of batches in training data: {train_len}")
print(f"Total number of batches in testing data: {test_len}")
print(f"Total number of batches in validation data: {validation_len}")

In [ ]:
px.bar(
    y=[train_len, test_len, validation_len],
    x=["Training", "Testing", "Validation"],
    title="Number of batches",
    text_auto=".2f"
)

In [ ]:
px.pie(
    values=[train_len, test_len, validation_len],
    names=["Training", "Testing", "Validation"],
    title="Percentage of batches by type",
)

## Optimize dataset pipeline for efficient training

In [ ]:
train_ds = batch_train.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
validation_ds = batch_valid.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds = batch_test.cache().shuffle(1000).prefetch(buffer_size=tf.data.AUTOTUNE)

## Define data augmentation pipeline

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1),
])

## Apply data augmentation to training dataset

In [ ]:
train_ds = train_ds.map(lambda X, y: (data_augmentation(X, training=True), y)).prefetch(buffer_size=tf.data.AUTOTUNE)

# Model Training

## Model Architecture

The model uses transfer learning with MobileNetV2 pretrained on ImageNet.

Components:

- Data Augmentation Layer
- Resizing Layer
- MobileNetV2 Backbone (frozen)
- Global Average Pooling Layer
- Dropout Layer
- Dense Hidden Layer (128 units, ReLU)
- Output Softmax Layer

Purpose:
- MobileNetV2 extracts high-level image features
- Global Average Pooling reduces feature dimensions
- Dropout helps prevent overfitting
- Dense layers perform disease classification

In [ ]:
input_shape = (IMAGE_SIZE, IMAGE_SIZE, CHANNEL)

# Load pre-trained MobileNetV2 backbone

base_model = MobileNetV2(
    input_shape=input_shape,
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = models.Sequential([
    data_augmentation,
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

model.build((None, IMAGE_SIZE, IMAGE_SIZE, CHANNEL))
model.summary()

### Configure training callbacks

In [ ]:
# Stop training when validation loss stops improving

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Save the best-performing model

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    model_path,
    save_best_only=True,
    monitor='val_accuracy'
)

# Reduce learning rate when validation loss plateaus

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3
)

### Compiling the model

In [ ]:
model.compile(optimizer='adam', loss=tf.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])

history = model.fit(
    train_ds,
    validation_data=validation_ds,
    verbose=1,
    callbacks=[early_stopping, checkpoint, reduce_lr],
    epochs=EPOCHS
)

## Fine Tuning

### Setting callbacks for fine tuning

In [ ]:
# Stop training when validation loss stops improving

early_stopping_ft = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# Save the best-performing model

checkpoint_ft = tf.keras.callbacks.ModelCheckpoint(
    "best_model_finetuned.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# Reduce learning rate when validation loss plateaus

reduce_lr_ft = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    verbose=1
)

### Compiling the model

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping_ft,checkpoint_ft,reduce_lr_ft]
)

## Evaluate model performance on the test set

In [ ]:
loss, accuracy = model.evaluate(test_ds)

print(f"Test Accuracy: {accuracy:.4f}")

## Save the model

In [ ]:
model.save(filepath=model_path)
joblib.dump(history.history, filename=status_path)

# Model Evaluation

## Confusion Matrix

In [ ]:
y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)

    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

cf = confusion_matrix(y_true, y_pred)

px.imshow(
    cf,
    title="Confusion Matrix",
    text_auto=".2f"
)

## Inspect training history and recorded metrics

In [ ]:
print(history)
print(history.params)
print(history.history.keys())

## Inspect recorded loss metrics

In [ ]:
history.history['loss'][:5]

## Extract training and validation metrics

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

## Visualize training and validation performance

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

epochs = list(range(EPOCHS))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Training and Validation Accuracy",
                    "Training and Validation Loss")
)

# Accuracy
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=acc,
        mode='lines',
        name='Training Accuracy'
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=epochs,
        y=val_acc,
        mode='lines',
        name='Validation Accuracy'
    ),
    row=1,
    col=1
)

# Loss
fig.add_trace(
    go.Scatter(
        x=epochs,
        y=loss,
        mode='lines',
        name='Training Loss'
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=epochs,
        y=val_loss,
        mode='lines',
        name='Validation Loss'
    ),
    row=1,
    col=2
)

fig.update_layout(
    height=500,
    width=1000,
    title_text="Model Training Metrics",
    hovermode="x unified"
)

fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_xaxes(title_text="Epoch", row=1, col=2)

fig.update_yaxes(title_text="Accuracy", row=1, col=1)
fig.update_yaxes(title_text="Loss", row=1, col=2)

fig.show()

## Visualize a sample prediction

In [ ]:
for images_batch, labels_batch in test_ds.take(1):
    first_image = images_batch[0].numpy().astype("uint8")
    first_label = labels_batch[0].numpy()

    batch_prediction = model.predict(images_batch, verbose=0)
    predicted_label = class_names[np.argmax(batch_prediction[0])]

    print("Actual Label:", class_names[first_label])
    print("Predicted Label:", predicted_label)

    fig = go.Figure(
        go.Image(z=first_image)
    )

    confidence = np.max(batch_prediction[0]) * 100

    fig.update_layout(
        title=(
            f"Actual: {class_names[first_label]}<br>"
            f"Predicted: {predicted_label} ({confidence:.2f}%)"
        )
    )

    fig.show()

    break

## Sample Predictions

In [ ]:
def predict(model, img):
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array, verbose=0)

    predicted_class = class_names[np.argmax(predictions[0])]
    confidence = round(100 * np.max(predictions[0]), 2)

    return predicted_class, confidence

In [ ]:
for images, labels in test_ds.take(1):
    titles = []

    for i in range(9):
        predicted_class, confidence = predict(
            model,
            images[i].numpy()
        )
        actual_class = class_names[int(labels[i])]
        symbol = "✅" if predicted_class == actual_class else "❌"
        titles.append(
            f"{symbol} {actual_class}<br>"
            f"{predicted_class}<br>"
            f"{confidence:.1f}%"
        )

    fig = make_subplots(
        rows=3,
        cols=3,
        subplot_titles=titles,
        horizontal_spacing=0.03,
        vertical_spacing=0.12
    )

    for i in range(9):
        row = i // 3 + 1
        col = i % 3 + 1
        fig.add_trace(
            go.Image(
                z=images[i].numpy().astype("uint8")
            ),
            row=row,
            col=col
        )

    fig.update_layout(
        height=1000,
        width=1000,
        showlegend=False,
    )

    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.show()

    break

## Saving the variables

In [ ]:
variables = {
    "image_size": IMAGE_SIZE,
    "channel": CHANNEL,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "training_len": train_len,
    "testing_len": test_len,
    "validation_len": validation_len,
    "y_true": y_true,
    "y_pred": y_pred,
    "model_name": "MobileNetV2",
    "transfer_learning": True,
    "fine_tuned_layers": 20,
    "learning_rate": 1e-5,
    "classification_report": classification_report(y_true,y_pred,output_dict=True)
}

joblib.dump(variables, variables_path)